# Foundations

In [0]:
from pyspark.sql import Row

# Setup for SQL and PySpark foundation exercises
history_data = [
    Row(user_id="u001", track_id="t100", play_duration_sec=180),
    Row(user_id="u002", track_id="t101", play_duration_sec=210),
    Row(user_id="u001", track_id="t101", play_duration_sec=45),
    Row(user_id="u003", track_id="t102", play_duration_sec=300),
    Row(user_id="u002", track_id="t100", play_duration_sec=190),
    Row(user_id="u004", track_id="t100", play_duration_sec=None)
]
tracks_data = [
    Row(track_id="t100", artist_name="The Pythons", genre="Rock"),
    Row(track_id="t101", artist_name="SQL Syndicate", genre="Electronic"),
    Row(track_id="t102", artist_name="Data Bricks", genre="Rock")
]

df_history = spark.createDataFrame(history_data)
df_tracks = spark.createDataFrame(tracks_data)

df_history.createOrReplaceTempView("listening_history")
df_tracks.createOrReplaceTempView("tracks")

##Phase 1: Python Essentials
We are focusing on raw Python structures and API requests. Do this in a standard Python cell.

Exercise 1.1: The API Call. Use the requests library to make a GET request to this public placeholder API endpoint: https://jsonplaceholder.typicode.com/users. Parse the JSON response and assign it to a variable called raw_api_data.



In [0]:
import requests, json
raw_api_data = requests.get('https://jsonplaceholder.typicode.com/users').json()

print(f"Sucess! We have a {type(raw_api_data)} of data")

"""Printing list extyracted for the JSON not easy to see structure
for listValue in raw_api_data:
    print(listValue)
""" 

# Using json.dumps() to make the resulting JSON string human-readable by adding line breaks and two spaces of indentation per nesting level.
formatted_data = json.dumps(raw_api_data, indent=2)
print(formatted_data)

Exercise 1.2: Dictionaries & Lists. Write a function named extract_user_cities(data) that takes your raw_api_data (which is a list of dictionaries) as an argument. The function must iterate through the data and return a standard Python list containing only the names of the cities those users live in.

In [0]:
def extract_user_cities(data):
    user_cities = []
    for user in data:
        user_cities.append(user["address"]["city"])
    return user_cities

user_cities = extract_user_cities(raw_api_data)
print(user_cities)

## Phase 2: The SQL Bedrock
Your coach said no advanced stuff, just mastery of the core. Use %sql cells in Databricks for these. You will query the listening_history and tracks views you created in the setup cell.

Exercise 2.1: The Core Join & Aggregate. Write a SQL query that returns the artist_name and the total combined play_duration_sec for all of their tracks. You will need to join the two tables and group the results.

In [0]:
%sql
with cte(
  select track_id, sum(play_duration_sec) as total_play_time 
  from listening_history group by track_id
) select t.artist_name, c.total_play_time from tracks t left join cte c on t.track_id = c.track_id


In [0]:
%sql
select * from listening_history limit 10

In [0]:
%sql
select * from tracks limit 10;

In [0]:
%sql
SELECT 
    t.artist_name, 
    SUM(l.play_duration_sec) AS total_play_time 
FROM tracks t 
JOIN listening_history l 
    ON t.track_id = l.track_id
GROUP BY t.artist_name;

Exercise 2.2: Filtering and Shaping. Write a SQL query to find out how many unique users listened to the "Rock" genre.

In [0]:
%sql
select count(distinct(user_id)) as unique_user_rock 
from listening_history l join tracks t on l.track_id = t.track_id 
where t.genre = 'Rock'

## Phase 3: PySpark Fundamentals
PySpark is built for scale. Time to use the DataFrame API using the df_history and df_tracks dataframes generated in the setup cell. Use standard Python cells for this.

Exercise 3.1: Safe Fetching. Take df_history and create a new dataframe called clean_history. Filter out any rows where play_duration_sec is null (because bad data blows up pipelines).



In [0]:
import pyspark.sql.functions as F
df_clean_history = df_history.filter(F.col("play_duration_sec").isNotNull())
df_clean_history.display()
#display(df_clean_history)

Exercise 3.2: PySpark Aggregation. Using your new clean_history dataframe, group the data by user_id, calculate the average play_duration_sec per user, and rename that new aggregated column to avg_listen_time. Finally, display the results using Databricks' built-in display() function.

In [0]:
df_clean_history.groupBy(F.col("user_id"))\
    .agg(F.avg("play_duration_sec").alias("avg_play_duration"))\
        .display()